# Binary trees in OCaml: a lab you can run, change and watch

**How to use this notebook in class.** Every section has the same rhythm:

1. **The code**: a small function, shown in full.
2. **The picture and the output**: real trees drawn from real OCaml values, and test tables that run the code live.
3. **Change it**: the same cell contains (or is followed by) a *planted bug*, a one-line change to the code. The test table shows which cases flip from ✓ to ✗, so students *see* what the change did.

Every picture and every table in this notebook is produced by the OCaml code you see, not pasted in.

**Reading the pictures.** Blue circles are `Node`s. An orange badge is the order in which a traversal visits a node. Orange or green nodes mark a search or insertion path. In the pictures that show `Leaf`, small grey squares are empty subtrees.

**Reading the tables.** Green `✓` means the function returned what the *expected* column says. Red `✗` means it did not, and shows what it returned instead. The last row counts the passes.

**To re-run a demo with your own change:** edit the function in its cell, run the cell, then run the table cell again.

| Section | What it shows |
|---|---|
| 1 | The `tree` type, and seven fixture trees |
| 2 | `size`, `height`, `leaves`, `sum`, each with a planted bug |
| 3 | Four traversals (pre, in, post, level order), and why `@` makes `preorder` slow |
| 4 | `tree_fold`: one function behind all of the above, and the accumulator trap |
| 5 | `map`, `mirror`, `filter` (two meanings) on trees |
| 6 | Binary search trees: `mem`, `insert`, `remove` (three cases), and the invariant |
| 7 | Balance: sorted input, rotations, and AVL trees |
| 8 | Randomized testing against a model, with a failing input shrunk to a minimum |

In [86]:
#use "topfind";;
#require "jupyter.notebook";;

- : unit = ()
Findlib has been successfully loaded. Additional directives:
  #require "package";;      to load a package
  #list;;                   to list the available packages
  #camlp4o;;                to load camlp4 (standard syntax)
  #camlp4r;;                to load camlp4 (revised syntax)
  #predicates "p,q,...";;   to set these predicates
  Topfind.reset();;         to force that packages will be reloaded
  #thread;;                 to enable threads

- : unit = ()


## Toolkit (run once, no need to read)

The tree type, a drawing function that turns any `int tree` into SVG, and a test-table function. Everything after this cell is the actual lesson.

In [87]:
type 'a tree = Leaf | Node of 'a * 'a tree * 'a tree

let leaf v = Node (v, Leaf, Leaf)

(* compact text form: 4[2[1,3],6[5,7]]; a node with no children is just its value, ∅ is an empty tree *)
let rec sexp = function
  | Leaf -> "∅"
  | Node (v, Leaf, Leaf) -> string_of_int v
  | Node (v, l, r) -> Printf.sprintf "%d[%s,%s]" v (sexp l) (sexp r)

let show_list l = "[" ^ String.concat "; " (List.map string_of_int l) ^ "]"

(* ---------- drawing ---------- *)
let css = {|<style>
.tl{border-collapse:collapse;font-family:ui-monospace,Menlo,monospace;font-size:13px;margin:6px 0}
.tl th{background:#1f3a5f;color:#fff;padding:4px 10px;text-align:left}
.tl td{text-align:left}
.tl td{border:1px solid #c9d1d9;padding:3px 10px;color:#111;background:#fff}
.tl td.ok{background:#e6f4ea;color:#0b6b2b}
.tl td.bad{background:#fde7e9;color:#a4141b;font-weight:bold}
.tl td.sum{background:#f1f3f5;font-weight:bold}
.tf{display:flex;flex-wrap:wrap;gap:14px;align-items:flex-start}
.tf figure{margin:0;text-align:center;font:13px sans-serif;color:#222;background:#fff;border:1px solid #d0d7de;border-radius:6px;padding:6px 8px}
.tf figcaption{margin-top:4px}
.tt{font:600 14px sans-serif;margin:6px 0 2px 0;color:#111;background:#fff;padding:2px 4px}
</style>|}

type style = { fill : string; badge : string option }
let plain _ = { fill = "#dbe9ff"; badge = None }

let svg_of ?(leaves = false) ?(style = plain) (t : int tree) =
  let dx = 40 and dy = 56 and r = 17 and m = 14 in
  let counter = ref 0 and items = ref [] and edges = ref [] in
  let rec go d = function
    | Leaf ->
        if leaves then begin
          let x = !counter in incr counter; items := (None, x, d) :: !items; Some x
        end else None
    | Node (v, l, rt) ->
        let xl = go (d + 1) l in
        let x = !counter in incr counter;
        items := (Some v, x, d) :: !items;
        let xr = go (d + 1) rt in
        List.iter (function Some cx -> edges := (x, d, cx, d + 1) :: !edges | None -> ()) [ xl; xr ];
        Some x
  in
  ignore (go 0 t);
  if !items = [] then {|<div style="font:13px sans-serif;color:#555;padding:8px 14px;background:#fff">∅ (empty tree)</div>|}
  else begin
    let maxd = List.fold_left (fun a (_, _, d) -> max a d) 0 !items in
    let px x = m + r + x * dx and py d = m + r + d * dy in
    let w = 2 * (m + r) + (max 1 !counter - 1) * dx and h = 2 * (m + r) + maxd * dy in
    let b = Buffer.create 1024 in
    Printf.bprintf b {|<svg xmlns="http://www.w3.org/2000/svg" width="%d" height="%d" font-family="sans-serif" font-size="14" style="background:#fff">|} w h;
    List.iter (fun (x1, d1, x2, d2) ->
        Printf.bprintf b {|<line x1="%d" y1="%d" x2="%d" y2="%d" stroke="#6b7785" stroke-width="1.5"/>|}
          (px x1) (py d1) (px x2) (py d2)) !edges;
    List.iter (fun (v, x, d) ->
        match v with
        | None -> Printf.bprintf b {|<rect x="%d" y="%d" width="10" height="10" fill="#c9d1d9"/>|} (px x - 5) (py d - 5)
        | Some v ->
            let s = style v in
            Printf.bprintf b {|<circle cx="%d" cy="%d" r="%d" fill="%s" stroke="#27496d" stroke-width="1.5"/><text x="%d" y="%d" text-anchor="middle" fill="#111">%d</text>|}
              (px x) (py d) r s.fill (px x) (py d + 5) v;
            (match s.badge with
             | Some bd ->
                 Printf.bprintf b {|<circle cx="%d" cy="%d" r="9" fill="#d93f0b"/><text x="%d" y="%d" text-anchor="middle" font-size="11" fill="#fff">%s</text>|}
                   (px x + r - 2) (py d - r + 2) (px x + r - 2) (py d - r + 6) bd
             | None -> ()))
      (List.rev !items);
    Buffer.add_string b "</svg>";
    Buffer.contents b
  end

(* the short sleep lets the kernel flush each picture before the cell ends; without it the last output of a cell can be dropped when a notebook is run from the command line *)
let show_html s = ignore (Jupyter_notebook.display "text/html" (css ^ s)); Unix.sleepf 0.05
let figure caption svg = Printf.sprintf "<figure>%s<figcaption>%s</figcaption></figure>" svg caption
let draw ?leaves ?style caption t = (caption, svg_of ?leaves ?style t)
let show_trees items =
  show_html ("<div class='tf'>" ^ String.concat "" (List.map (fun (c, s) -> figure c s) items) ^ "</div>")

let index_of x l =
  let rec go i = function [] -> None | y :: ys -> if y = x then Some i else go (i + 1) ys in
  go 1 l
let numbered vs v =
  match index_of v vs with Some i -> { fill = "#ffe3c2"; badge = Some (string_of_int i) } | None -> plain v
let on_path vs v = if List.mem v vs then { fill = "#ffd6a5"; badge = None } else plain v
let found_at path target v =
  if v = target && List.mem v path then { fill = "#b7e4c7"; badge = None } else on_path path v

(* ---------- test tables ---------- *)
let esc s =
  String.concat "" (List.map (function '<' -> "&lt;" | '>' -> "&gt;" | '&' -> "&amp;" | c -> String.make 1 c)
                      (List.init (String.length s) (String.get s)))

let show_table ?(title = "") headers rows =
  let b = Buffer.create 1024 in
  if title <> "" then Printf.bprintf b "<div class='tt'>%s</div>" (esc title);
  Buffer.add_string b "<table class='tl'><tr>";
  List.iter (fun h -> Printf.bprintf b "<th>%s</th>" (esc h)) headers;
  Buffer.add_string b "</tr>";
  List.iter (fun row ->
      Buffer.add_string b "<tr>";
      List.iter (fun c -> Printf.bprintf b "<td>%s</td>" (esc c)) row;
      Buffer.add_string b "</tr>") rows;
  Buffer.add_string b "</table>";
  show_html (Buffer.contents b)

let run_cases ?(title = "") ~show_in ~show_out cases impls =
  let b = Buffer.create 2048 in
  if title <> "" then Printf.bprintf b "<div class='tt'>%s</div>" (esc title);
  Buffer.add_string b "<table class='tl'><tr><th>case</th><th>input</th><th>expected</th>";
  List.iter (fun (n, _) -> Printf.bprintf b "<th>%s</th>" (esc n)) impls;
  Buffer.add_string b "</tr>";
  let passed = Array.make (List.length impls) 0 in
  List.iter (fun (name, input, expected) ->
      Printf.bprintf b "<tr><td>%s</td><td>%s</td><td>%s</td>" (esc name) (esc (show_in input)) (esc (show_out expected));
      List.iteri (fun i (_, f) ->
          match (try Ok (f input) with e -> Error (Printexc.to_string e)) with
          | Ok got when got = expected ->
              passed.(i) <- passed.(i) + 1;
              Printf.bprintf b "<td class='ok'>✓ %s</td>" (esc (show_out got))
          | Ok got -> Printf.bprintf b "<td class='bad'>✗ %s</td>" (esc (show_out got))
          | Error e -> Printf.bprintf b "<td class='bad'>✗ raised %s</td>" (esc e)) impls;
      Buffer.add_string b "</tr>") cases;
  Printf.bprintf b "<tr><td class='sum' colspan='3'>passed</td>";
  Array.iter (fun p -> Printf.bprintf b "<td class='sum'>%d / %d</td>" p (List.length cases)) passed;
  Buffer.add_string b "</tr></table>";
  show_html (Buffer.contents b)

type 'a tree = Leaf | Node of 'a * 'a tree * 'a tree


val leaf : 'a -> 'a tree = <fun>


val sexp : int tree -> string = <fun>


val show_list : int list -> string = <fun>


val css : string =
  "<style>\n.tl{border-collapse:collapse;font-family:ui-monospace,Menlo,monospace;font-size:13px;margin:6px 0}\n.tl th{background:#1f3a5f;color:#fff;padding:4px 10px;text-align:left}\n.tl td{text-align:left}\n.tl td{border:1px solid #c9d1d9;padding:3px 10px;color:#111;background:#fff}\n.tl td.ok{background"... (* string length 775; truncated *)


...

val on_path : 'a list -> 'a -> style = <fun>


val found_at : 'a list -> 'a -> 'a -> style = <fun>


val esc : string -> string = <fun>


val show_table : ?title:string -> string list -> string list list -> unit =
  <fun>


val run_cases :
  ?title:string ->
  show_in:('a -> string) ->
  show_out:('b -> string) ->
  (string * 'a * 'b) list -> (string * ('a -> 'b)) list -> unit = <fun>


## 1. The type, and the trees we will use all lab

```ocaml
type 'a tree = Leaf | Node of 'a * 'a tree * 'a tree
```

A tree is **either empty (`Leaf`) or a `Node` holding a value and two subtrees**. Every function in this notebook is a `match` on those two cases. *Say this out loud once: the shape of the code is the shape of the type.*

Careful with the vocabulary: `Leaf` is an **empty subtree**. A *leaf node* in the textbook sense, a node with no children, is `Node (v, Leaf, Leaf)`. The next cell draws the grey squares so you can see the difference.

In [88]:
let empty : int tree = Leaf
let single = leaf 5
let perfect7 = Node (4, Node (2, leaf 1, leaf 3), Node (6, leaf 5, leaf 7))
let ragged = Node (4, leaf 2, Node (6, leaf 5, Node (9, Leaf, leaf 10)))        (* the tree from the Part 2 activity *)
let chain_left = Node (4, Node (3, Node (2, leaf 1, Leaf), Leaf), Leaf)
let chain_right = Node (1, Leaf, Node (2, Leaf, Node (3, Leaf, Node (4, Leaf, leaf 5))))
let not_bst = Node (5, Node (3, leaf 1, leaf 8), leaf 7)                          (* looks fine locally; 8 is in the wrong place *)

let fixtures = [ ("empty", empty); ("single", single); ("perfect7", perfect7); ("ragged", ragged);
                 ("chain_left", chain_left); ("chain_right", chain_right); ("not_bst", not_bst) ]

val empty : int tree = Leaf


val single : int tree = Node (5, Leaf, Leaf)


val perfect7 : int tree =
  Node (4, Node (2, Node (1, Leaf, Leaf), Node (3, Leaf, Leaf)),
   Node (6, Node (5, Leaf, Leaf), Node (7, Leaf, Leaf)))


val ragged : int tree =
  Node (4, Node (2, Leaf, Leaf),
   Node (6, Node (5, Leaf, Leaf), Node (9, Leaf, Node (10, Leaf, Leaf))))


val chain_left : int tree =
  Node (4, Node (3, Node (2, Node (1, Leaf, Leaf), Leaf), Leaf), Leaf)


val chain_right : int tree =
  Node (1, Leaf,
   Node (2, Leaf, Node (3, Leaf, Node (4, Leaf, Node (5, Leaf, Leaf)))))


val not_bst : int tree =
  Node (5, Node (3, Node (1, Leaf, Leaf), Node (8, Leaf, Leaf)),
   Node (7, Leaf, Leaf))


val fixtures : (string * int tree) list =
  [("empty", Leaf); ("single", Node (5, Leaf, Leaf));
   ("perfect7",
    Node (4, Node (2, Node (1, Leaf, Leaf), Node (3, Leaf, Leaf)),
     Node (6, Node (5, Leaf, Leaf), Node (7, Leaf, Leaf))));
   ("ragged",
    Node (4, Node (2, Leaf, Leaf),
     Node (6, Node (5, Leaf, Leaf), Node (9, Leaf, Node (10, Leaf, Leaf)))));
   ("chain_left",
    Node (4, Node (3, Node (2, Node (1, Leaf, Leaf), Leaf), Leaf), Leaf));
   ("chain_right",
    Node (1, Leaf,
     Node (2, Leaf, Node (3, Leaf, Node (4, Leaf, Node (5, Leaf, Leaf))))));
   ("not_bst",
    Node (5, Node (3, Node (1, Leaf, Leaf), Node (8, Leaf, Leaf)),
     Node (7, Leaf, Leaf)))]


In [89]:
let () = show_trees (List.map (fun (n, t) -> draw n t) fixtures)

∅ (empty tree) empty 5 single 1 2 3 4 5 6 7 perfect7 2 4 5 6 9 10 ragged 1 2 3 4 chain_left 1 2 3 4 5 chain_right 1 3 8 5 7 not_bst

The same two trees with the empty subtrees drawn as grey squares. Count them: a tree with *n* nodes always has *n + 1* of them. That fact is why a careless `leaves` function is off by one (section 2).

In [90]:
let () = show_trees [ draw ~leaves:true "perfect7, with its empty subtrees" perfect7;
                      draw ~leaves:true "ragged: node 9 has no left child" ragged ]

1 2 3 4 5 6 7 perfect7, with its empty subtrees 2 4 5 6 9 10 ragged: node 9 has no left child

## 2. Measuring a tree: `size`, `height`, `leaves`, `sum`

Each of these is the same recursion: answer for `Leaf`, then combine the answers from the two subtrees.

In [91]:
let rec size = function Leaf -> 0 | Node (_, l, r) -> 1 + size l + size r
let rec height = function Leaf -> 0 | Node (_, l, r) -> 1 + max (height l) (height r)
let rec leaves = function Leaf -> 0 | Node (_, Leaf, Leaf) -> 1 | Node (_, l, r) -> leaves l + leaves r
let rec sum = function Leaf -> 0 | Node (v, l, r) -> v + sum l + sum r

val size : 'a tree -> int = <fun>


val height : 'a tree -> int = <fun>


val leaves : 'a tree -> int = <fun>


val sum : int tree -> int = <fun>


**Predict, then run.** The expected numbers below are typed by hand (that is what a student predicts), and the real functions are then run against them.

In [92]:
let cases expected = List.map2 (fun (n, t) e -> (n, t, e)) fixtures expected

let () =
  run_cases ~title:"size" ~show_in:sexp ~show_out:string_of_int (cases [ 0; 1; 7; 6; 4; 5; 5 ]) [ ("size", size) ];
  run_cases ~title:"height (the number of nodes on the longest path)" ~show_in:sexp ~show_out:string_of_int
    (cases [ 0; 1; 3; 4; 4; 5; 3 ]) [ ("height", height) ];
  run_cases ~title:"leaves (nodes with no children)" ~show_in:sexp ~show_out:string_of_int
    (cases [ 0; 1; 4; 3; 1; 1; 3 ]) [ ("leaves", leaves) ];
  run_cases ~title:"sum" ~show_in:sexp ~show_out:string_of_int (cases [ 0; 5; 28; 36; 10; 15; 24 ]) [ ("sum", sum) ]

val cases : 'a list -> (string * int tree * 'a) list = <fun>


size case input expected size empty ∅ 0 ✓ 0 single 5 1 ✓ 1 perfect7 4[2[1,3],6[5,7]] 7 ✓ 7 ragged 4[2,6[5,9[∅,10]]] 6 ✓ 6 chain_left 4[3[2[1,∅],∅],∅] 4 ✓ 4 chain_right 1[∅,2[∅,3[∅,4[∅,5]]]] 5 ✓ 5 not_bst 5[3[1,8],7] 5 ✓ 5 passed 7 / 7

height (the number of nodes on the longest path) case input expected height empty ∅ 0 ✓ 0 single 5 1 ✓ 1 perfect7 4[2[1,3],6[5,7]] 3 ✓ 3 ragged 4[2,6[5,9[∅,10]]] 4 ✓ 4 chain_left 4[3[2[1,∅],∅],∅] 4 ✓ 4 chain_right 1[∅,2[∅,3[∅,4[∅,5]]]] 5 ✓ 5 not_bst 5[3[1,8],7] 3 ✓ 3 passed 7 / 7

leaves (nodes with no children) case input expected leaves empty ∅ 0 ✓ 0 single 5 1 ✓ 1 perfect7 4[2[1,3],6[5,7]] 4 ✓ 4 ragged 4[2,6[5,9[∅,10]]] 3 ✓ 3 chain_left 4[3[2[1,∅],∅],∅] 1 ✓ 1 chain_right 1[∅,2[∅,3[∅,4[∅,5]]]] 1 ✓ 1 not_bst 5[3[1,8],7] 3 ✓ 3 passed 7 / 7

sum case input expected sum empty ∅ 0 ✓ 0 single 5 5 ✓ 5 perfect7 4[2[1,3],6[5,7]] 28 ✓ 28 ragged 4[2,6[5,9[∅,10]]] 36 ✓ 36 chain_left 4[3[2[1,∅],∅],∅] 10 ✓ 10 chain_right 1[∅,2[∅,3[∅,4[∅,5]]]] 15 ✓ 15 not_bst 5[3[1,8],7] 24 ✓ 24 passed 7 / 7

### Change it: four one-line bugs

Each function below differs from the real one by **one small change**. Run the table and look at *which rows* go red. The pattern of failures tells you what the change did.

| Bug | The change |
|---|---|
| `height_bug` | the base case returns `1` instead of `0` |
| `height_min` | `min` instead of `max` |
| `size_bug` | forgets to count the right subtree |
| `leaves_bug` | the base case returns `1` and the "no children" case is gone, so it counts the empty subtrees |

In [93]:
let rec height_bug = function Leaf -> 1 | Node (_, l, r) -> 1 + max (height_bug l) (height_bug r)
let rec height_min = function Leaf -> 0 | Node (_, l, r) -> 1 + min (height_min l) (height_min r)
let rec size_bug = function Leaf -> 0 | Node (_, l, _) -> 1 + size_bug l
let rec leaves_bug = function Leaf -> 1 | Node (_, l, r) -> leaves_bug l + leaves_bug r

let () =
  run_cases ~title:"height: the real one against two bugs" ~show_in:sexp ~show_out:string_of_int
    (cases [ 0; 1; 3; 4; 4; 5; 3 ]) [ ("height", height); ("height_bug", height_bug); ("height_min", height_min) ];
  run_cases ~title:"size: forgetting the right subtree" ~show_in:sexp ~show_out:string_of_int
    (cases [ 0; 1; 7; 6; 4; 5; 5 ]) [ ("size", size); ("size_bug", size_bug) ];
  run_cases ~title:"leaves: counting empty subtrees" ~show_in:sexp ~show_out:string_of_int
    (cases [ 0; 1; 4; 3; 1; 1; 3 ]) [ ("leaves", leaves); ("leaves_bug", leaves_bug) ]

val height_bug : 'a tree -> int = <fun>


val height_min : 'a tree -> int = <fun>


val size_bug : 'a tree -> int = <fun>


val leaves_bug : 'a tree -> int = <fun>


height: the real one against two bugs case input expected height height_bug height_min empty ∅ 0 ✓ 0 ✗ 1 ✓ 0 single 5 1 ✓ 1 ✗ 2 ✓ 1 perfect7 4[2[1,3],6[5,7]] 3 ✓ 3 ✗ 4 ✓ 3 ragged 4[2,6[5,9[∅,10]]] 4 ✓ 4 ✗ 5 ✗ 2 chain_left 4[3[2[1,∅],∅],∅] 4 ✓ 4 ✗ 5 ✗ 1 chain_right 1[∅,2[∅,3[∅,4[∅,5]]]] 5 ✓ 5 ✗ 6 ✗ 1 not_bst 5[3[1,8],7] 3 ✓ 3 ✗ 4 ✗ 2 passed 7 / 7 0 / 7 3 / 7

size: forgetting the right subtree case input expected size size_bug empty ∅ 0 ✓ 0 ✓ 0 single 5 1 ✓ 1 ✓ 1 perfect7 4[2[1,3],6[5,7]] 7 ✓ 7 ✗ 3 ragged 4[2,6[5,9[∅,10]]] 6 ✓ 6 ✗ 2 chain_left 4[3[2[1,∅],∅],∅] 4 ✓ 4 ✓ 4 chain_right 1[∅,2[∅,3[∅,4[∅,5]]]] 5 ✓ 5 ✗ 1 not_bst 5[3[1,8],7] 5 ✓ 5 ✗ 3 passed 7 / 7 3 / 7

leaves: counting empty subtrees case input expected leaves leaves_bug empty ∅ 0 ✓ 0 ✗ 1 single 5 1 ✓ 1 ✗ 2 perfect7 4[2[1,3],6[5,7]] 4 ✓ 4 ✗ 8 ragged 4[2,6[5,9[∅,10]]] 3 ✓ 3 ✗ 7 chain_left 4[3[2[1,∅],∅],∅] 1 ✓ 1 ✗ 5 chain_right 1[∅,2[∅,3[∅,4[∅,5]]]] 1 ✓ 1 ✗ 6 not_bst 5[3[1,8],7] 3 ✓ 3 ✗ 6 passed 7 / 7 0 / 7

**Things to point out.**
- `height_min` is right on `empty`, `single` and `perfect7`, and wrong on everything lopsided. A *perfect* tree hides the bug, because its shortest and longest paths are the same length. *Test cases should include lopsided trees.*
- `size_bug` passes on `chain_left`, because a left chain never needed the right subtree.
- `leaves_bug` is off by exactly one on every non-empty tree. Look back at the grey squares: *n + 1* empty subtrees.

## 3. Four ways to walk a tree

| Order | Visit |
|---|---|
| pre-order | the node, then left, then right |
| in-order | left, then the node, then right |
| post-order | left, then right, then the node |
| level-order | by depth, left to right (breadth first) |

In [94]:
let rec preorder = function Leaf -> [] | Node (v, l, r) -> [ v ] @ preorder l @ preorder r
let rec inorder = function Leaf -> [] | Node (v, l, r) -> inorder l @ [ v ] @ inorder r
let rec postorder = function Leaf -> [] | Node (v, l, r) -> postorder l @ postorder r @ [ v ]

let levelorder t =
  let rec go level =
    let vals = List.filter_map (function Leaf -> None | Node (v, _, _) -> Some v) level in
    if vals = [] then []
    else vals @ go (List.concat_map (function Leaf -> [] | Node (_, l, r) -> [ l; r ]) level)
  in
  go [ t ]

val preorder : 'a tree -> 'a list = <fun>


val inorder : 'a tree -> 'a list = <fun>


val postorder : 'a tree -> 'a list = <fun>


val levelorder : 'a tree -> 'a list = <fun>


The badge on each node is its position in the visit order. Same tree, four different numberings:

In [95]:
let () =
  show_trees (List.map (fun (name, walk) -> draw ~style:(numbered (walk ragged)) name ragged)
                [ ("pre-order", preorder); ("in-order", inorder); ("post-order", postorder); ("level-order", levelorder) ]);
  show_trees (List.map (fun (name, walk) -> draw ~style:(numbered (walk perfect7)) name perfect7)
                [ ("pre-order", preorder); ("in-order", inorder); ("post-order", postorder); ("level-order", levelorder) ])

2 2 4 1 5 4 6 3 9 5 10 6 pre-order 2 1 4 2 5 3 6 4 9 5 10 6 in-order 2 1 4 6 5 2 6 5 9 4 10 3 post-order 2 2 4 1 5 4 6 3 9 5 10 6 level-order

1 3 2 2 3 4 4 1 5 6 6 5 7 7 pre-order 1 1 2 2 3 3 4 4 5 5 6 6 7 7 in-order 1 1 2 3 3 2 4 7 5 4 6 6 7 5 post-order 1 4 2 2 3 5 4 1 5 6 6 3 7 7 level-order

In [96]:
let walks_expected =
  [ ("empty", empty, [ []; []; []; [] ]);
    ("single", single, [ [ 5 ]; [ 5 ]; [ 5 ]; [ 5 ] ]);
    ("perfect7", perfect7, [ [ 4; 2; 1; 3; 6; 5; 7 ]; [ 1; 2; 3; 4; 5; 6; 7 ]; [ 1; 3; 2; 5; 7; 6; 4 ]; [ 4; 2; 6; 1; 3; 5; 7 ] ]);
    ("ragged", ragged, [ [ 4; 2; 6; 5; 9; 10 ]; [ 2; 4; 5; 6; 9; 10 ]; [ 2; 5; 10; 9; 6; 4 ]; [ 4; 2; 6; 5; 9; 10 ] ]);
    ("chain_left", chain_left, [ [ 4; 3; 2; 1 ]; [ 1; 2; 3; 4 ]; [ 1; 2; 3; 4 ]; [ 4; 3; 2; 1 ] ]);
    ("chain_right", chain_right, [ [ 1; 2; 3; 4; 5 ]; [ 1; 2; 3; 4; 5 ]; [ 5; 4; 3; 2; 1 ]; [ 1; 2; 3; 4; 5 ] ]);
    ("not_bst", not_bst, [ [ 5; 3; 1; 8; 7 ]; [ 1; 3; 8; 5; 7 ]; [ 1; 8; 3; 7; 5 ]; [ 5; 3; 7; 1; 8 ] ]) ]

let walk_cases k = List.map (fun (n, t, e) -> (n, t, List.nth e k)) walks_expected

let () =
  run_cases ~title:"pre-order" ~show_in:sexp ~show_out:show_list (walk_cases 0) [ ("preorder", preorder) ];
  run_cases ~title:"in-order" ~show_in:sexp ~show_out:show_list (walk_cases 1) [ ("inorder", inorder) ];
  run_cases ~title:"post-order" ~show_in:sexp ~show_out:show_list (walk_cases 2) [ ("postorder", postorder) ];
  run_cases ~title:"level-order" ~show_in:sexp ~show_out:show_list (walk_cases 3) [ ("levelorder", levelorder) ]

val walks_expected : (string * int tree * int list list) list =
  [("empty", Leaf, [[]; []; []; []]);
   ("single", Node (5, Leaf, Leaf), [[5]; [5]; [5]; [5]]);
   ("perfect7",
    Node (4, Node (2, Node (1, Leaf, Leaf), Node (3, Leaf, Leaf)),
     Node (6, Node (5, Leaf, Leaf), Node (7, Leaf, Leaf))),
    [[4; 2; 1; 3; 6; 5; 7]; [1; 2; 3; 4; 5; 6; 7]; [1; 3; 2; 5; 7; 6; 4];
     [4; 2; 6; 1; 3; 5; 7]]);
   ("ragged",
    Node (4, Node (2, Leaf, Leaf),
     Node (6, Node (5, Leaf, Leaf), Node (9, Leaf, Node (10, Leaf, Leaf)))),
    [[4; 2; 6; 5; 9; 10]; [2; 4; 5; 6; 9; 10]; [2; 5; 10; 9; 6; 4];
     [4; 2; 6; 5; 9; 10]]);
   ("chain_left",
    Node (4, Node (3, Node (2, Node (1, Leaf, Leaf), Leaf), Leaf), Leaf),
    [[4; 3; 2; 1]; [1; 2; 3; 4]; [1; 2; 3; 4]; [4; 3; 2; 1]]);
   ("chain_right",
    Node (1, Leaf,
     Node (2, Leaf, Node (3, Leaf, Node (4, Leaf, Node (5, Leaf, Leaf))))),
    [[1; 2; 3; 4; 5]; [1; 2; 3; 4; 5]; [5; 4; 3; 2; 1]; [1; 2; 3; 4; 5]]);
   ("not_bst",
    Node (5

val walk_cases : int -> (string * int tree * int list) list = <fun>


pre-order case input expected preorder empty ∅ [] ✓ [] single 5 [5] ✓ [5] perfect7 4[2[1,3],6[5,7]] [4; 2; 1; 3; 6; 5; 7] ✓ [4; 2; 1; 3; 6; 5; 7] ragged 4[2,6[5,9[∅,10]]] [4; 2; 6; 5; 9; 10] ✓ [4; 2; 6; 5; 9; 10] chain_left 4[3[2[1,∅],∅],∅] [4; 3; 2; 1] ✓ [4; 3; 2; 1] chain_right 1[∅,2[∅,3[∅,4[∅,5]]]] [1; 2; 3; 4; 5] ✓ [1; 2; 3; 4; 5] not_bst 5[3[1,8],7] [5; 3; 1; 8; 7] ✓ [5; 3; 1; 8; 7] passed 7 / 7

in-order case input expected inorder empty ∅ [] ✓ [] single 5 [5] ✓ [5] perfect7 4[2[1,3],6[5,7]] [1; 2; 3; 4; 5; 6; 7] ✓ [1; 2; 3; 4; 5; 6; 7] ragged 4[2,6[5,9[∅,10]]] [2; 4; 5; 6; 9; 10] ✓ [2; 4; 5; 6; 9; 10] chain_left 4[3[2[1,∅],∅],∅] [1; 2; 3; 4] ✓ [1; 2; 3; 4] chain_right 1[∅,2[∅,3[∅,4[∅,5]]]] [1; 2; 3; 4; 5] ✓ [1; 2; 3; 4; 5] not_bst 5[3[1,8],7] [1; 3; 8; 5; 7] ✓ [1; 3; 8; 5; 7] passed 7 / 7

post-order case input expected postorder empty ∅ [] ✓ [] single 5 [5] ✓ [5] perfect7 4[2[1,3],6[5,7]] [1; 3; 2; 5; 7; 6; 4] ✓ [1; 3; 2; 5; 7; 6; 4] ragged 4[2,6[5,9[∅,10]]] [2; 5; 10; 9; 6; 4] ✓ [2; 5; 10; 9; 6; 4] chain_left 4[3[2[1,∅],∅],∅] [1; 2; 3; 4] ✓ [1; 2; 3; 4] chain_right 1[∅,2[∅,3[∅,4[∅,5]]]] [5; 4; 3; 2; 1] ✓ [5; 4; 3; 2; 1] not_bst 5[3[1,8],7] [1; 8; 3; 7; 5] ✓ [1; 8; 3; 7; 5] passed 7 / 7

level-order case input expected levelorder empty ∅ [] ✓ [] single 5 [5] ✓ [5] perfect7 4[2[1,3],6[5,7]] [4; 2; 6; 1; 3; 5; 7] ✓ [4; 2; 6; 1; 3; 5; 7] ragged 4[2,6[5,9[∅,10]]] [4; 2; 6; 5; 9; 10] ✓ [4; 2; 6; 5; 9; 10] chain_left 4[3[2[1,∅],∅],∅] [4; 3; 2; 1] ✓ [4; 3; 2; 1] chain_right 1[∅,2[∅,3[∅,4[∅,5]]]] [1; 2; 3; 4; 5] ✓ [1; 2; 3; 4; 5] not_bst 5[3[1,8],7] [5; 3; 7; 1; 8] ✓ [5; 3; 7; 1; 8] passed 7 / 7

### Change it: swap two operands and you have a different traversal

`inorder_rev` swaps the two recursive calls. `pre_as_post` moves `[v]` to the end. Run the tables and look at which orders each one *accidentally* matches.

In [97]:
let rec inorder_rev = function Leaf -> [] | Node (v, l, r) -> inorder_rev r @ [ v ] @ inorder_rev l
let rec pre_as_post = function Leaf -> [] | Node (v, l, r) -> pre_as_post l @ pre_as_post r @ [ v ]

let () =
  run_cases ~title:"in-order against inorder_rev (swapped operands)" ~show_in:sexp ~show_out:show_list
    (walk_cases 1) [ ("inorder", inorder); ("inorder_rev", inorder_rev) ];
  run_cases ~title:"pre-order against pre_as_post (the [v] moved to the end)" ~show_in:sexp ~show_out:show_list
    (walk_cases 0) [ ("preorder", preorder); ("pre_as_post", pre_as_post) ]

val inorder_rev : 'a tree -> 'a list = <fun>


val pre_as_post : 'a tree -> 'a list = <fun>


in-order against inorder_rev (swapped operands) case input expected inorder inorder_rev empty ∅ [] ✓ [] ✓ [] single 5 [5] ✓ [5] ✓ [5] perfect7 4[2[1,3],6[5,7]] [1; 2; 3; 4; 5; 6; 7] ✓ [1; 2; 3; 4; 5; 6; 7] ✗ [7; 6; 5; 4; 3; 2; 1] ragged 4[2,6[5,9[∅,10]]] [2; 4; 5; 6; 9; 10] ✓ [2; 4; 5; 6; 9; 10] ✗ [10; 9; 6; 5; 4; 2] chain_left 4[3[2[1,∅],∅],∅] [1; 2; 3; 4] ✓ [1; 2; 3; 4] ✗ [4; 3; 2; 1] chain_right 1[∅,2[∅,3[∅,4[∅,5]]]] [1; 2; 3; 4; 5] ✓ [1; 2; 3; 4; 5] ✗ [5; 4; 3; 2; 1] not_bst 5[3[1,8],7] [1; 3; 8; 5; 7] ✓ [1; 3; 8; 5; 7] ✗ [7; 5; 8; 3; 1] passed 7 / 7 2 / 7

pre-order against pre_as_post (the [v] moved to the end) case input expected preorder pre_as_post empty ∅ [] ✓ [] ✓ [] single 5 [5] ✓ [5] ✓ [5] perfect7 4[2[1,3],6[5,7]] [4; 2; 1; 3; 6; 5; 7] ✓ [4; 2; 1; 3; 6; 5; 7] ✗ [1; 3; 2; 5; 7; 6; 4] ragged 4[2,6[5,9[∅,10]]] [4; 2; 6; 5; 9; 10] ✓ [4; 2; 6; 5; 9; 10] ✗ [2; 5; 10; 9; 6; 4] chain_left 4[3[2[1,∅],∅],∅] [4; 3; 2; 1] ✓ [4; 3; 2; 1] ✗ [1; 2; 3; 4] chain_right 1[∅,2[∅,3[∅,4[∅,5]]]] [1; 2; 3; 4; 5] ✓ [1; 2; 3; 4; 5] ✗ [5; 4; 3; 2; 1] not_bst 5[3[1,8],7] [5; 3; 1; 8; 7] ✓ [5; 3; 1; 8; 7] ✗ [1; 8; 3; 7; 5] passed 7 / 7 2 / 7

**Things to point out.**
- `inorder_rev` passes on `empty` and `single`, *and nothing else*. It really is a descending in-order walk. On a binary search tree it lists the values **largest first**.
- `chain_left` and `chain_right` are the interesting ones: with no left subtree, in-order and pre-order give the same answer, and with no right subtree, in-order and post-order do. Different codes, same output on some trees: the test set has to be varied enough to tell them apart.

### Why `@` makes `preorder` slow

`[v] @ preorder l @ preorder r` copies the left list on every level. On a left-leaning chain of *n* nodes that is about *n²/2* list cells. The accumulator version builds the answer with one `::` per node.

In [98]:
let preorder_acc t =
  let rec go acc = function Leaf -> acc | Node (v, l, r) -> v :: go (go acc r) l in
  go [] t

let rec left_chain n = if n = 0 then Leaf else Node (n, left_chain (n - 1), Leaf)
let time f = let t0 = Sys.time () in ignore (f ()); (Sys.time () -. t0) *. 1000.0

let () =
  show_table ~title:"milliseconds to run preorder on a left chain of n nodes"
    [ "n"; "preorder (with @)"; "preorder_acc (with ::)"; "same answer?" ]
    (List.map (fun n ->
         let t = left_chain n in
         [ string_of_int n; Printf.sprintf "%.0f ms" (time (fun () -> preorder t));
           Printf.sprintf "%.1f ms" (time (fun () -> preorder_acc t));
           string_of_bool (preorder t = preorder_acc t) ]) [ 2000; 4000; 8000; 16000 ])

val preorder_acc : 'a tree -> 'a list = <fun>


val left_chain : int -> int tree = <fun>


val time : (unit -> 'a) -> float = <fun>


n,preorder (with @),preorder_acc (with ::),same answer?
2000,19 ms,0.0 ms,true
4000,79 ms,0.1 ms,true
8000,349 ms,0.1 ms,true
16000,1681 ms,0.3 ms,true


Double *n* and the `@` version takes four to five times as long, while the accumulator version takes about twice as long. (The exact milliseconds depend on the machine. The ratio is the lesson.)

## 4. One function behind them all: `tree_fold`

`tree_fold op acc t` replaces every `Leaf` with `acc` and every `Node (v, l, r)` with `op v (answer for l) (answer for r)`.

In [99]:
let rec tree_fold op acc = function
  | Leaf -> acc
  | Node (v, l, r) -> op v (tree_fold op acc l) (tree_fold op acc r)

let size_f t = tree_fold (fun _ a b -> 1 + a + b) 0 t
let height_f t = tree_fold (fun _ a b -> 1 + max a b) 0 t
let sum_f t = tree_fold (fun v a b -> v + a + b) 0 t
let inorder_f t = tree_fold (fun v a b -> a @ [ v ] @ b) [] t
let preorder_f t = tree_fold (fun v a b -> (v :: a) @ b) [] t

val tree_fold : ('a -> 'b -> 'b -> 'b) -> 'b -> 'a tree -> 'b = <fun>


val size_f : 'a tree -> int = <fun>


val height_f : 'a tree -> int = <fun>


val sum_f : int tree -> int = <fun>


val inorder_f : 'a tree -> 'a list = <fun>


val preorder_f : 'a tree -> 'a list = <fun>


In [100]:
let () =
  run_cases ~title:"the fold versions agree with the hand-written ones" ~show_in:sexp ~show_out:string_of_int
    (cases [ 0; 1; 7; 6; 4; 5; 5 ]) [ ("size", size); ("size_f", size_f) ];
  run_cases ~show_in:sexp ~show_out:show_list (walk_cases 1) [ ("inorder", inorder); ("inorder_f", inorder_f) ];
  run_cases ~show_in:sexp ~show_out:show_list (walk_cases 0) [ ("preorder", preorder); ("preorder_f", preorder_f) ]

the fold versions agree with the hand-written ones case input expected size size_f empty ∅ 0 ✓ 0 ✓ 0 single 5 1 ✓ 1 ✓ 1 perfect7 4[2[1,3],6[5,7]] 7 ✓ 7 ✓ 7 ragged 4[2,6[5,9[∅,10]]] 6 ✓ 6 ✓ 6 chain_left 4[3[2[1,∅],∅],∅] 4 ✓ 4 ✓ 4 chain_right 1[∅,2[∅,3[∅,4[∅,5]]]] 5 ✓ 5 ✓ 5 not_bst 5[3[1,8],7] 5 ✓ 5 ✓ 5 passed 7 / 7 7 / 7

case input expected inorder inorder_f empty ∅ [] ✓ [] ✓ [] single 5 [5] ✓ [5] ✓ [5] perfect7 4[2[1,3],6[5,7]] [1; 2; 3; 4; 5; 6; 7] ✓ [1; 2; 3; 4; 5; 6; 7] ✓ [1; 2; 3; 4; 5; 6; 7] ragged 4[2,6[5,9[∅,10]]] [2; 4; 5; 6; 9; 10] ✓ [2; 4; 5; 6; 9; 10] ✓ [2; 4; 5; 6; 9; 10] chain_left 4[3[2[1,∅],∅],∅] [1; 2; 3; 4] ✓ [1; 2; 3; 4] ✓ [1; 2; 3; 4] chain_right 1[∅,2[∅,3[∅,4[∅,5]]]] [1; 2; 3; 4; 5] ✓ [1; 2; 3; 4; 5] ✓ [1; 2; 3; 4; 5] not_bst 5[3[1,8],7] [1; 3; 8; 5; 7] ✓ [1; 3; 8; 5; 7] ✓ [1; 3; 8; 5; 7] passed 7 / 7 7 / 7

case input expected preorder preorder_f empty ∅ [] ✓ [] ✓ [] single 5 [5] ✓ [5] ✓ [5] perfect7 4[2[1,3],6[5,7]] [4; 2; 1; 3; 6; 5; 7] ✓ [4; 2; 1; 3; 6; 5; 7] ✓ [4; 2; 1; 3; 6; 5; 7] ragged 4[2,6[5,9[∅,10]]] [4; 2; 6; 5; 9; 10] ✓ [4; 2; 6; 5; 9; 10] ✓ [4; 2; 6; 5; 9; 10] chain_left 4[3[2[1,∅],∅],∅] [4; 3; 2; 1] ✓ [4; 3; 2; 1] ✓ [4; 3; 2; 1] chain_right 1[∅,2[∅,3[∅,4[∅,5]]]] [1; 2; 3; 4; 5] ✓ [1; 2; 3; 4; 5] ✓ [1; 2; 3; 4; 5] not_bst 5[3[1,8],7] [5; 3; 1; 8; 7] ✓ [5; 3; 1; 8; 7] ✓ [5; 3; 1; 8; 7] passed 7 / 7 7 / 7

### Change it: the starting value is part of the answer

`acc` is what the fold returns for an **empty** tree. Build a product with `acc = 1` and with `acc = 0`:

In [101]:
let product_ok t = tree_fold (fun v a b -> v * a * b) 1 t
let product_bad t = tree_fold (fun v a b -> v * a * b) 0 t

let () =
  run_cases ~title:"the product of all values" ~show_in:sexp ~show_out:string_of_int
    (cases [ 1; 5; 5040; 21600; 24; 120; 840 ]) [ ("acc = 1", product_ok); ("acc = 0", product_bad) ]

val product_ok : int tree -> int = <fun>


val product_bad : int tree -> int = <fun>


the product of all values case input expected acc = 1 acc = 0 empty ∅ 1 ✓ 1 ✗ 0 single 5 5 ✓ 5 ✗ 0 perfect7 4[2[1,3],6[5,7]] 5040 ✓ 5040 ✗ 0 ragged 4[2,6[5,9[∅,10]]] 21600 ✓ 21600 ✗ 0 chain_left 4[3[2[1,∅],∅],∅] 24 ✓ 24 ✗ 0 chain_right 1[∅,2[∅,3[∅,4[∅,5]]]] 120 ✓ 120 ✗ 0 not_bst 5[3[1,8],7] 840 ✓ 840 ✗ 0 passed 7 / 7 0 / 7

With `acc = 0`, every `Leaf` hands a `0` to its parent, the parent multiplies it in, and the zero climbs to the root. Even the *empty* tree fails: the product of nothing is `1`, just as the sum of nothing is `0`. (**The identity element of the operation.**)

### Change it again: the lightest path, and the `0` that wins a `min`

The lightest path from the root to a node with no children. Using `0` as the empty-subtree answer works for `max` (all our values are positive) but silently breaks `min`:

In [102]:
let heavy t = tree_fold (fun v a b -> v + max a b) 0 t
let light_bad t = tree_fold (fun v a b -> v + min a b) 0 t
let light_ok t =
  tree_fold (fun v a b ->
      match a, b with
      | None, None -> Some v
      | Some x, None | None, Some x -> Some (v + x)
      | Some x, Some y -> Some (v + min x y)) None t

let () =
  run_cases ~title:"heaviest path: acc = 0 is safe for max (values are positive)" ~show_in:sexp ~show_out:string_of_int
    (cases [ 0; 5; 17; 29; 10; 15; 16 ]) [ ("heavy", heavy) ];
  run_cases ~title:"lightest path to a childless node" ~show_in:sexp
    ~show_out:(function Some n -> string_of_int n | None -> "None")
    (cases [ None; Some 5; Some 7; Some 6; Some 10; Some 15; Some 9 ])
    [ ("light_ok", light_ok); ("light_bad", fun t -> match t with Leaf -> None | _ -> Some (light_bad t)) ]

val heavy : int tree -> int = <fun>


val light_bad : int tree -> int = <fun>


val light_ok : int tree -> int option = <fun>


heaviest path: acc = 0 is safe for max (values are positive) case input expected heavy empty ∅ 0 ✓ 0 single 5 5 ✓ 5 perfect7 4[2[1,3],6[5,7]] 17 ✓ 17 ragged 4[2,6[5,9[∅,10]]] 29 ✓ 29 chain_left 4[3[2[1,∅],∅],∅] 10 ✓ 10 chain_right 1[∅,2[∅,3[∅,4[∅,5]]]] 15 ✓ 15 not_bst 5[3[1,8],7] 16 ✓ 16 passed 7 / 7

lightest path to a childless node case input expected light_ok light_bad empty ∅ None ✓ None ✓ None single 5 5 ✓ 5 ✓ 5 perfect7 4[2[1,3],6[5,7]] 7 ✓ 7 ✓ 7 ragged 4[2,6[5,9[∅,10]]] 6 ✓ 6 ✓ 6 chain_left 4[3[2[1,∅],∅],∅] 10 ✓ 10 ✗ 4 chain_right 1[∅,2[∅,3[∅,4[∅,5]]]] 15 ✓ 15 ✗ 1 not_bst 5[3[1,8],7] 9 ✓ 9 ✓ 9 passed 7 / 7 5 / 7

`light_bad` is right on `single`, `perfect7`, `ragged` and `not_bst`, and wrong on both chains. In a chain every node has an empty side, so `min` picks the `0` for "a path that stops here", and the path ends at a node that still has a child. The accumulator was doing arithmetic it was never meant to do.

## 5. Transforming trees: `map`, `mirror`, `filter`

In [103]:
let rec tree_map f = function Leaf -> Leaf | Node (v, l, r) -> Node (f v, tree_map f l, tree_map f r)
let rec mirror = function Leaf -> Leaf | Node (v, l, r) -> Node (v, mirror r, mirror l)

let () =
  show_trees [ draw "ragged" ragged;
               draw "tree_map (fun x -> x * x) ragged" (tree_map (fun x -> x * x) ragged);
               draw "mirror ragged" (mirror ragged) ]

val tree_map : ('a -> 'b) -> 'a tree -> 'b tree = <fun>


val mirror : 'a tree -> 'a tree = <fun>


2 4 5 6 9 10 ragged 4 16 25 36 81 100 tree_map (fun x -> x * x) ragged 10 9 6 5 4 2 mirror ragged

Two ways to "filter" a tree, and they answer different questions. **`filter_prune`** removes a failing node *and everything below it*. **`filter_keep`** keeps the values that pass, in order, as a list.

In [104]:
let rec filter_prune p = function
  | Leaf -> Leaf
  | Node (v, l, r) -> if p v then Node (v, filter_prune p l, filter_prune p r) else Leaf

let rec filter_keep p = function
  | Leaf -> []
  | Node (v, l, r) -> filter_keep p l @ (if p v then [ v ] else []) @ filter_keep p r

let not6 v = v <> 6
let () =
  show_trees [ draw "ragged" ragged; draw "filter_prune (<> 6) ragged: the whole right side is gone" (filter_prune not6 ragged) ];
  show_html ("<div class='tt'>filter_keep (&lt;&gt; 6) ragged = " ^ esc (show_list (filter_keep not6 ragged)) ^ "</div>")

val filter_prune : ('a -> bool) -> 'a tree -> 'a tree = <fun>


val filter_keep : ('a -> bool) -> 'a tree -> 'a list = <fun>


val not6 : int -> bool = <fun>


2 4 5 6 9 10 ragged 2 4 filter_prune (<> 6) ragged: the whole right side is gone

filter_keep (<> 6) ragged = [2; 4; 5; 9; 10]

`filter_prune (<> 6)` loses **5, 9 and 10**, which pass the test, because their ancestor `6` does not. `filter_keep` loses only `6`. The test below makes the difference a number:

In [105]:
let () =
  run_cases ~title:"how many values survive `<> 6`?" ~show_in:sexp ~show_out:string_of_int
    [ ("ragged", ragged, 5); ("perfect7", perfect7, 6); ("chain_left", chain_left, 4) ]
    [ ("size (filter_prune ..)", fun t -> size (filter_prune not6 t));
      ("length (filter_keep ..)", fun t -> List.length (filter_keep not6 t)) ]

how many values survive `<> 6`? case input expected size (filter_prune ..) length (filter_keep ..) ragged 4[2,6[5,9[∅,10]]] 5 ✗ 2 ✓ 5 perfect7 4[2[1,3],6[5,7]] 6 ✗ 4 ✓ 6 chain_left 4[3[2[1,∅],∅],∅] 4 ✓ 4 ✓ 4 passed 1 / 3 3 / 3

### A property worth checking: `mirror` reverses the in-order walk

`inorder (mirror t) = List.rev (inorder t)`, for every tree. This is a *property*: a statement that should hold on every input, so every fixture is a test.

In [106]:
let () =
  run_cases ~title:"inorder (mirror t) = List.rev (inorder t)" ~show_in:sexp ~show_out:string_of_bool
    (List.map (fun (n, t) -> (n, t, true)) fixtures)
    [ ("holds?", fun t -> inorder (mirror t) = List.rev (inorder t)) ]

inorder (mirror t) = List.rev (inorder t) case input expected holds? empty ∅ true ✓ true single 5 true ✓ true perfect7 4[2[1,3],6[5,7]] true ✓ true ragged 4[2,6[5,9[∅,10]]] true ✓ true chain_left 4[3[2[1,∅],∅],∅] true ✓ true chain_right 1[∅,2[∅,3[∅,4[∅,5]]]] true ✓ true not_bst 5[3[1,8],7] true ✓ true passed 7 / 7

## 6. Binary *search* trees

A binary search tree adds one rule, the **invariant**: for every node, **every** value in the left subtree is smaller and **every** value in the right subtree is larger. The rule is what lets `mem` ignore half the tree at every step.

In [107]:
let rec mem x = function
  | Leaf -> false
  | Node (v, l, r) -> if x < v then mem x l else if x > v then mem x r else true

let rec mem_any x = function                     (* searches the whole tree; no invariant needed *)
  | Leaf -> false
  | Node (v, l, r) -> v = x || mem_any x l || mem_any x r

let rec insert x = function
  | Leaf -> leaf x
  | Node (v, l, r) as t -> if x < v then Node (v, insert x l, r) else if x > v then Node (v, l, insert x r) else t

let rec path x = function                        (* the values visited on the way to x *)
  | Leaf -> []
  | Node (v, l, r) -> v :: (if x < v then path x l else if x > v then path x r else [])

let bst_of_list l = List.fold_left (fun t x -> insert x t) Leaf l

val mem : 'a -> 'a tree -> bool = <fun>


val mem_any : 'a -> 'a tree -> bool = <fun>


val insert : 'a -> 'a tree -> 'a tree = <fun>


val path : 'a -> 'a tree -> 'a list = <fun>


val bst_of_list : 'a list -> 'a tree = <fun>


### Searching: watch the path

Orange nodes are the ones `mem` looked at. Green is where it found the key. Notice how little of the tree it touches.

In [108]:
let () =
  show_trees [ draw ~style:(found_at (path 5 ragged) 5) "mem 5: found after 3 looks" ragged;
               draw ~style:(found_at (path 8 ragged) 8) "mem 8: falls off at the empty left of 9" ragged;
               draw ~style:(found_at (path 10 ragged) 10) "mem 10: 4 looks" ragged ];
  show_table [ "call"; "path"; "result" ]
    (List.map (fun x -> [ Printf.sprintf "mem %d ragged" x; show_list (path x ragged); string_of_bool (mem x ragged) ]) [ 5; 8; 10; 1 ])

2 4 5 6 9 10 mem 5: found after 3 looks 2 4 5 6 9 10 mem 8: falls off at the empty left of 9 2 4 5 6 9 10 mem 10: 4 looks

call,path,result
mem 5 ragged,[4; 6; 5],true
mem 8 ragged,[4; 6; 9],false
mem 10 ragged,[4; 6; 9; 10],true
mem 1 ragged,[4; 2],false


### Inserting: a new leaf at the end of the search path

`insert` follows the same path as `mem` and hangs the new value where the search would have fallen off. Inserting a value that is already there returns the tree unchanged.

In [109]:
let () =
  show_trees [ draw "before" ragged;
               draw ~style:(found_at (path 8 (insert 8 ragged)) 8) "insert 8 (green is the new node)" (insert 8 ragged);
               draw ~style:(found_at (path 0 (insert 0 ragged)) 0) "insert 0" (insert 0 ragged);
               draw "insert 6 (already there)" (insert 6 ragged) ]

2 4 5 6 9 10 before 2 4 5 6 8 9 10 insert 8 (green is the new node) 0 2 4 5 6 9 10 insert 0 2 4 5 6 9 10 insert 6 (already there)

### The invariant has to be checked against *every* ancestor, not just the parent

`not_bst` passes the parent check at every node: `3 < 5`, `1 < 3`, `8 > 3`, `7 > 5`. But `8` sits in the left subtree of `5`. The correct check carries a lower and an upper bound down the tree.

In [110]:
let is_bst t =
  let rec ok lo hi = function
    | Leaf -> true
    | Node (v, l, r) ->
        (match lo with Some b -> v > b | None -> true) && (match hi with Some b -> v < b | None -> true)
        && ok lo (Some v) l && ok (Some v) hi r
  in
  ok None None t

let rec is_bst_naive = function        (* only compares each node with its two children *)
  | Leaf -> true
  | Node (v, l, r) ->
      (match l with Node (a, _, _) -> a < v | Leaf -> true) && (match r with Node (b, _, _) -> b > v | Leaf -> true)
      && is_bst_naive l && is_bst_naive r

let () =
  run_cases ~title:"is the tree a binary search tree?" ~show_in:sexp ~show_out:string_of_bool
    [ ("empty", empty, true); ("single", single, true); ("perfect7", perfect7, true); ("ragged", ragged, true);
      ("chain_left", chain_left, true); ("chain_right", chain_right, true);
      ("not_bst", not_bst, false); ("mirror perfect7", mirror perfect7, false) ]
    [ ("is_bst", is_bst); ("is_bst_naive", is_bst_naive) ]

val is_bst : 'a tree -> bool = <fun>


val is_bst_naive : 'a tree -> bool = <fun>


is the tree a binary search tree? case input expected is_bst is_bst_naive empty ∅ true ✓ true ✓ true single 5 true ✓ true ✓ true perfect7 4[2[1,3],6[5,7]] true ✓ true ✓ true ragged 4[2,6[5,9[∅,10]]] true ✓ true ✓ true chain_left 4[3[2[1,∅],∅],∅] true ✓ true ✓ true chain_right 1[∅,2[∅,3[∅,4[∅,5]]]] true ✓ true ✓ true not_bst 5[3[1,8],7] false ✓ false ✗ true mirror perfect7 4[6[7,5],2[3,1]] false ✓ false ✓ false passed 8 / 8 7 / 8

### Why the invariant matters: `mem` on a tree that breaks it

`mem` trusts the invariant. On `not_bst` it looks for `8`, sees `8 > 5`, goes right, and never visits the left subtree where `8` actually is.

In [111]:
let () =
  run_cases ~title:"mem 8 on not_bst: the BST search against a search of the whole tree" ~show_in:(fun (x, t) -> Printf.sprintf "mem %d (%s)" x (sexp t))
    ~show_out:string_of_bool
    [ ("8 in not_bst", (8, not_bst), true); ("3 in not_bst", (3, not_bst), true); ("7 in not_bst", (7, not_bst), true);
      ("8 in perfect7", (8, perfect7), false); ("5 in perfect7", (5, perfect7), true) ]
    [ ("mem (uses the invariant)", fun (x, t) -> mem x t); ("mem_any (whole tree)", fun (x, t) -> mem_any x t) ]

mem 8 on not_bst: the BST search against a search of the whole tree case input expected mem (uses the invariant) mem_any (whole tree) 8 in not_bst mem 8 (5[3[1,8],7]) true ✗ false ✓ true 3 in not_bst mem 3 (5[3[1,8],7]) true ✓ true ✓ true 7 in not_bst mem 7 (5[3[1,8],7]) true ✓ true ✓ true 8 in perfect7 mem 8 (4[2[1,3],6[5,7]]) false ✓ false ✓ false 5 in perfect7 mem 5 (4[2[1,3],6[5,7]]) true ✓ true ✓ true passed 4 / 5 5 / 5

### Change it: flip one comparison in `insert`

`insert_bug` sends smaller values right and larger values left. It still runs and still builds trees. The trees are just *mirror images*, so `mem` and `is_bst` both fail.

In [112]:
let rec insert_bug x = function
  | Leaf -> leaf x
  | Node (v, l, r) as t -> if x > v then Node (v, insert_bug x l, r) else if x < v then Node (v, l, insert_bug x r) else t

let () =
  let items = [ 4; 2; 6; 1; 3; 5; 7 ] in
  show_trees [ draw "insert (correct)" (List.fold_left (fun t x -> insert x t) Leaf items);
               draw "insert_bug (> and < flipped)" (List.fold_left (fun t x -> insert_bug x t) Leaf items) ];
  run_cases ~title:"build a tree from [4;2;6;1;3;5;7]: is it a BST, and does mem find all seven values?" ~show_in:show_list ~show_out:string_of_bool
    [ ("is a BST and mem finds everything", items, true) ]
    [ ("correct insert", fun l -> let t = List.fold_left (fun t x -> insert x t) Leaf l in is_bst t && List.for_all (fun x -> mem x t) l);
      ("insert_bug", fun l -> let t = List.fold_left (fun t x -> insert_bug x t) Leaf l in is_bst t && List.for_all (fun x -> mem x t) l) ]

val insert_bug : 'a -> 'a tree -> 'a tree = <fun>


1 2 3 4 5 6 7 insert (correct) 7 6 5 4 3 2 1 insert_bug (> and < flipped)

build a tree from [4;2;6;1;3;5;7]: is it a BST, and does mem find all seven values? case input expected correct insert insert_bug is a BST and mem finds everything [4; 2; 6; 1; 3; 5; 7] true ✓ true ✗ false passed 1 / 1 0 / 1

### Removing: three cases

| The node being removed has | What to do |
|---|---|
| no children | replace it with `Leaf` |
| one child | replace it with that child |
| two children | replace its value with the **smallest value in its right subtree**, then remove that value from the right subtree |

The two-children case is the only hard one, and it is where students' bugs live.

In [113]:
let rec min_elt = function Leaf -> None | Node (v, Leaf, _) -> Some v | Node (_, l, _) -> min_elt l

let rec remove x = function
  | Leaf -> Leaf
  | Node (v, l, r) ->
      if x < v then Node (v, remove x l, r)
      else if x > v then Node (v, l, remove x r)
      else begin
        match l, r with
        | Leaf, _ -> r                               (* no left child: the right child (maybe Leaf) takes over *)
        | _, Leaf -> l                               (* one child on the left *)
        | _ -> (match min_elt r with                 (* two children *)
                | Some m -> Node (m, l, remove m r)
                | None -> l)
      end

val min_elt : 'a tree -> 'a option = <fun>


val remove : 'a -> 'a tree -> 'a tree = <fun>


In [114]:
let () =
  show_trees [ draw "ragged" ragged;
               draw "remove 10: a node with no children" (remove 10 ragged);
               draw "remove 9: one child (10 takes its place)" (remove 9 ragged);
               draw "remove 6: two children (smallest on the right, 9, moves up)" (remove 6 ragged);
               draw "remove 4: the root, two children (5 moves up)" (remove 4 ragged);
               draw "remove 7: not there, nothing changes" (remove 7 ragged) ]

2 4 5 6 9 10 ragged 2 4 5 6 9 remove 10: a node with no children 2 4 5 6 10 remove 9: one child (10 takes its place) 2 4 5 9 10 remove 6: two children (smallest on the right, 9, moves up) 2 5 6 9 10 remove 4: the root, two children (5 moves up) 2 4 5 6 9 10 remove 7: not there, nothing changes

In [115]:
(* the oracle: removing x from a BST should leave the sorted values minus x, and still be a BST *)
let remove_ok remove_fn (x, t) = inorder (remove_fn x t) = List.filter (fun v -> v <> x) (inorder t) && is_bst (remove_fn x t)

let () =
  run_cases ~title:"remove: the sorted values afterwards, and the invariant" ~show_in:(fun (x, t) -> Printf.sprintf "remove %d (%s)" x (sexp t))
    ~show_out:string_of_bool
    [ ("no children", (10, ragged), true); ("one child", (9, ragged), true); ("two children", (6, ragged), true);
      ("the root", (4, ragged), true); ("absent", (7, ragged), true); ("empty tree", (1, empty), true);
      ("perfect7 root", (4, perfect7), true); ("perfect7 inner", (2, perfect7), true) ]
    [ ("remove", remove_ok remove) ]

val remove_ok : ('a -> 'a tree -> 'a tree) -> 'a * 'a tree -> bool = <fun>


remove: the sorted values afterwards, and the invariant case input expected remove no children remove 10 (4[2,6[5,9[∅,10]]]) true ✓ true one child remove 9 (4[2,6[5,9[∅,10]]]) true ✓ true two children remove 6 (4[2,6[5,9[∅,10]]]) true ✓ true the root remove 4 (4[2,6[5,9[∅,10]]]) true ✓ true absent remove 7 (4[2,6[5,9[∅,10]]]) true ✓ true empty tree remove 1 (∅) true ✓ true perfect7 root remove 4 (4[2[1,3],6[5,7]]) true ✓ true perfect7 inner remove 2 (4[2[1,3],6[5,7]]) true ✓ true passed 8 / 8

### Change it: forget the two-children case

`remove_bug` handles every case except the third: when the node has two children it just keeps the left subtree, so **the right subtree disappears**.

In [116]:
let rec remove_bug x = function
  | Leaf -> Leaf
  | Node (v, l, r) ->
      if x < v then Node (v, remove_bug x l, r)
      else if x > v then Node (v, l, remove_bug x r)
      else (match l, r with Leaf, _ -> r | _, Leaf -> l | _ -> l)

let () =
  show_trees [ draw "remove 6 (correct)" (remove 6 ragged); draw "remove_bug 6: 9 and 10 vanished" (remove_bug 6 ragged) ];
  run_cases ~title:"remove against remove_bug" ~show_in:(fun (x, t) -> Printf.sprintf "remove %d (%s)" x (sexp t))
    ~show_out:string_of_bool
    [ ("no children", (10, ragged), true); ("one child", (9, ragged), true); ("two children", (6, ragged), true);
      ("the root", (4, ragged), true); ("absent", (7, ragged), true); ("perfect7 root", (4, perfect7), true);
      ("perfect7 inner", (2, perfect7), true); ("perfect7 leaf", (7, perfect7), true) ]
    [ ("remove", remove_ok remove); ("remove_bug", remove_ok remove_bug) ]

val remove_bug : 'a -> 'a tree -> 'a tree = <fun>


2 4 5 9 10 remove 6 (correct) 2 4 5 remove_bug 6: 9 and 10 vanished

remove against remove_bug case input expected remove remove_bug no children remove 10 (4[2,6[5,9[∅,10]]]) true ✓ true ✓ true one child remove 9 (4[2,6[5,9[∅,10]]]) true ✓ true ✓ true two children remove 6 (4[2,6[5,9[∅,10]]]) true ✓ true ✗ false the root remove 4 (4[2,6[5,9[∅,10]]]) true ✓ true ✗ false absent remove 7 (4[2,6[5,9[∅,10]]]) true ✓ true ✓ true perfect7 root remove 4 (4[2[1,3],6[5,7]]) true ✓ true ✗ false perfect7 inner remove 2 (4[2[1,3],6[5,7]]) true ✓ true ✗ false perfect7 leaf remove 7 (4[2[1,3],6[5,7]]) true ✓ true ✓ true passed 8 / 8 4 / 8

`remove_bug` passes exactly the cases that never reach the third branch. A test suite with only leaf removals and one-child removals would call this bug correct. *Every branch of a `match` needs a test that reaches it.*

### In-order is sorted, on a BST only

The in-order walk of a binary search tree is the sorted list. That is the invariant seen from outside, and it is why `bst_of_list` followed by `inorder` is a sort.

In [117]:
let () =
  show_table [ "tree"; "inorder"; "sorted?" ]
    (List.map (fun (n, t) -> let l = inorder t in [ n; show_list l; string_of_bool (l = List.sort compare l) ]) fixtures);
  show_html ("<div class='tt'>bst_of_list [5; 2; 8; 1; 9; 3; 5; 2] → inorder = "
             ^ esc (show_list (inorder (bst_of_list [ 5; 2; 8; 1; 9; 3; 5; 2 ]))) ^ " (sorted, duplicates dropped)</div>")

tree,inorder,sorted?
empty,[],true
single,[5],true
perfect7,[1; 2; 3; 4; 5; 6; 7],true
ragged,[2; 4; 5; 6; 9; 10],true
chain_left,[1; 2; 3; 4],true
chain_right,[1; 2; 3; 4; 5],true
not_bst,[1; 3; 8; 5; 7],false


bst_of_list [5; 2; 8; 1; 9; 3; 5; 2] → inorder = [1; 2; 3; 5; 8; 9] (sorted, duplicates dropped)

## 7. Balance: the shape decides the speed

`mem` follows one path, so its cost is the **height**, not the size. The same values can make a tree of height 3 or height 7.

In [118]:
let rec range a b = if a > b then [] else a :: range (a + 1) b

let shuffle l =
  let a = Array.of_list l in
  for i = Array.length a - 1 downto 1 do
    let j = Random.int (i + 1) in
    let t = a.(i) in a.(i) <- a.(j); a.(j) <- t
  done;
  Array.to_list a

let rec balanced_of_sorted = function            (* middle value as the root, recursively *)
  | [] -> Leaf
  | l ->
      let n = List.length l / 2 in
      let left = List.filteri (fun i _ -> i < n) l and right = List.filteri (fun i _ -> i > n) l in
      Node (List.nth l n, balanced_of_sorted left, balanced_of_sorted right)

let () =
  Random.init 3200;
  let vs = range 1 7 in
  show_trees [ draw "inserted in order 1..7" (bst_of_list vs);
               draw "inserted in a random order" (bst_of_list (shuffle vs));
               draw "built from the sorted list, middle first" (balanced_of_sorted vs) ]

val range : int -> int -> int list = <fun>


val shuffle : 'a list -> 'a list = <fun>


val balanced_of_sorted : 'a list -> 'a tree = <fun>


1 2 3 4 5 6 7 inserted in order 1..7 1 2 3 4 5 6 7 inserted in a random order 1 2 3 4 5 6 7 built from the sorted list, middle first

In [119]:
let () =
  Random.init 3200;
  show_table ~title:"height of a binary search tree holding 1..n"
    [ "n"; "inserted in order"; "inserted in random order"; "built balanced"; "log2(n+1), rounded up" ]
    (List.map (fun n ->
         let vs = range 1 n in
         [ string_of_int n; string_of_int (height (bst_of_list vs)); string_of_int (height (bst_of_list (shuffle vs)));
           string_of_int (height (balanced_of_sorted vs));
           string_of_int (int_of_float (ceil (log (float_of_int (n + 1)) /. log 2.0))) ])
       [ 7; 15; 63; 255; 1023; 4095 ])

n,inserted in order,inserted in random order,built balanced,"log2(n+1), rounded up"
7,7,5,3,3
15,15,6,4,4
63,63,11,6,6
255,255,14,8,8
1023,1023,22,10,10
4095,4095,30,12,12


Sorted input is the worst case: the "tree" is a linked list of height *n*. Random input lands in between, a small multiple of *log n*. Built balanced, it is exactly *log₂(n + 1)*. The difference in search cost:

In [120]:
let () =
  let n = 5000 in
  let vs = range 1 n in
  Random.init 3200;
  let degenerate = bst_of_list vs and balanced = balanced_of_sorted vs in
  let search_all t () = List.for_all (fun x -> mem x t) vs in
  show_table ~title:(Printf.sprintf "look up every value 1..%d once" n)
    [ "tree"; "height"; "time" ]
    [ [ "degenerate (inserted in order)"; string_of_int (height degenerate); Printf.sprintf "%.1f ms" (time (search_all degenerate)) ];
      [ "balanced"; string_of_int (height balanced); Printf.sprintf "%.2f ms" (time (search_all balanced)) ] ]

tree,height,time
degenerate (inserted in order),5000,226.5 ms
balanced,13,1.09 ms


### Rotations: a local rewrite that keeps the invariant

A **rotation** changes the shape of three nodes and leaves the in-order walk untouched. Look at the pictures: the in-order walk of each pair is the same list.

In [121]:
let rotate_right = function Node (y, Node (x, a, b), c) -> Node (x, a, Node (y, b, c)) | t -> t
let rotate_left = function Node (x, a, Node (y, b, c)) -> Node (y, Node (x, a, b), c) | t -> t

let () =
  let big = Node (8, Node (4, leaf 2, leaf 6), leaf 10) in
  show_trees [ draw "before" big; draw "rotate_right: 4 moves up" (rotate_right big) ];
  let big2 = Node (2, leaf 1, Node (6, leaf 4, leaf 8)) in
  show_trees [ draw "before" big2; draw "rotate_left: 6 moves up" (rotate_left big2) ];
  show_table [ "tree"; "in-order before"; "in-order after"; "equal?" ]
    [ [ "rotate_right"; show_list (inorder big); show_list (inorder (rotate_right big)); string_of_bool (inorder big = inorder (rotate_right big)) ];
      [ "rotate_left"; show_list (inorder big2); show_list (inorder (rotate_left big2)); string_of_bool (inorder big2 = inorder (rotate_left big2)) ] ]

val rotate_right : 'a tree -> 'a tree = <fun>


val rotate_left : 'a tree -> 'a tree = <fun>


2 4 6 8 10 before 2 4 6 8 10 rotate_right: 4 moves up

1 2 4 6 8 before 1 2 4 6 8 rotate_left: 6 moves up

tree,in-order before,in-order after,equal?
rotate_right,[2; 4; 6; 8; 10],[2; 4; 6; 8; 10],true
rotate_left,[1; 2; 4; 6; 8],[1; 2; 4; 6; 8],true


### AVL trees: rotate when a node gets lopsided

An **AVL tree** keeps every node *balanced*: the heights of its two subtrees differ by at most 1. After an insert, walk back up the path. At any node that has become lopsided, rotate. There are four shapes:

| Inserted into | Shape | Fix |
|---|---|---|
| left of the left child | left-left | one right rotation |
| right of the right child | right-right | one left rotation |
| right of the left child | left-right | rotate the child left, then rotate right |
| left of the right child | right-left | rotate the child right, then rotate left |

(This version recomputes `height` at every node, which is simple and slow. A real AVL tree stores the height in each node.)

In [122]:
let is_balanced t =
  let rec ok = function Leaf -> true | Node (_, l, r) -> abs (height l - height r) <= 1 && ok l && ok r in
  ok t

let rebalance = function
  | Leaf -> Leaf
  | Node (v, l, r) as t ->
      let hl = height l and hr = height r in
      if hl > hr + 1 then begin
        match l with
        | Node (_, ll, lr) when height ll < height lr -> rotate_right (Node (v, rotate_left l, r))   (* left-right *)
        | _ -> rotate_right t                                                                       (* left-left *)
      end else if hr > hl + 1 then begin
        match r with
        | Node (_, rl, rr) when height rr < height rl -> rotate_left (Node (v, l, rotate_right r))   (* right-left *)
        | _ -> rotate_left t                                                                        (* right-right *)
      end else t

let rec avl_insert x = function
  | Leaf -> leaf x
  | Node (v, l, r) as t ->
      if x < v then rebalance (Node (v, avl_insert x l, r))
      else if x > v then rebalance (Node (v, l, avl_insert x r))
      else t

let avl_of_list l = List.fold_left (fun t x -> avl_insert x t) Leaf l

val is_balanced : 'a tree -> bool = <fun>


val rebalance : 'a tree -> 'a tree = <fun>


val avl_insert : 'a -> 'a tree -> 'a tree = <fun>


val avl_of_list : 'a list -> 'a tree = <fun>


**The four cases, plain BST against AVL.** The same three values go in; the plain BST keeps the lopsided shape, the AVL tree rotates.

In [123]:
let () =
  List.iter (fun (name, vs) ->
      show_trees [ draw (name ^ ": BST") (bst_of_list vs); draw (name ^ ": AVL") (avl_of_list vs) ])
    [ ("left-left, insert 3, 2, 1", [ 3; 2; 1 ]); ("right-right, insert 1, 2, 3", [ 1; 2; 3 ]);
      ("left-right, insert 3, 1, 2", [ 3; 1; 2 ]); ("right-left, insert 1, 3, 2", [ 1; 3; 2 ]) ]

1 2 3 left-left, insert 3, 2, 1: BST 1 2 3 left-left, insert 3, 2, 1: AVL

1 2 3 right-right, insert 1, 2, 3: BST 1 2 3 right-right, insert 1, 2, 3: AVL

1 2 3 left-right, insert 3, 1, 2: BST 1 2 3 left-right, insert 3, 1, 2: AVL

1 2 3 right-left, insert 1, 3, 2: BST 1 2 3 right-left, insert 1, 3, 2: AVL

**Inserting 1, 2, 3, … 7 in order, one picture per insert.** The plain BST would be a chain of height 7.

In [124]:
let () =
  show_trees (List.map (fun k -> draw (Printf.sprintf "after inserting %d" k) (avl_of_list (range 1 k))) (range 1 7))

1 after inserting 1 1 2 after inserting 2 1 2 3 after inserting 3 1 2 3 4 after inserting 4 1 2 3 4 5 after inserting 5 1 2 3 4 5 6 after inserting 6 1 2 3 4 5 6 7 after inserting 7

### Change it: forget the double rotation

`rebalance_bug` always does the single rotation. That is correct for left-left and right-right, and **wrong for left-right and right-left**. Watch which insertion orders break.

In [125]:
let rebalance_bug = function
  | Leaf -> Leaf
  | Node (v, l, r) as t ->
      let hl = height l and hr = height r in
      if hl > hr + 1 then rotate_right t else if hr > hl + 1 then rotate_left t else t

let rec avl_insert_bug x = function
  | Leaf -> leaf x
  | Node (v, l, r) as t ->
      if x < v then rebalance_bug (Node (v, avl_insert_bug x l, r))
      else if x > v then rebalance_bug (Node (v, l, avl_insert_bug x r))
      else t

let build ins l = List.fold_left (fun t x -> ins x t) Leaf l
let good ins l = let t = build ins l in is_balanced t && is_bst t && inorder t = List.sort_uniq compare l

let () =
  show_trees [ draw "insert 3, 1, 2 (AVL)" (avl_of_list [ 3; 1; 2 ]);
               draw "insert 3, 1, 2 (rebalance_bug)" (build avl_insert_bug [ 3; 1; 2 ]) ];
  run_cases ~title:"is the result a balanced BST holding the right values?" ~show_in:show_list ~show_out:string_of_bool
    [ ("left-left", [ 3; 2; 1 ], true); ("right-right", [ 1; 2; 3 ], true); ("left-right", [ 3; 1; 2 ], true);
      ("right-left", [ 1; 3; 2 ], true); ("1..15 in order", range 1 15, true);
      ("15..1 in reverse", List.rev (range 1 15), true); ("zig-zag", [ 5; 1; 9; 2; 8; 3; 7; 4; 6 ], true) ]
    [ ("avl_insert", good avl_insert); ("avl_insert_bug", good avl_insert_bug) ]

val rebalance_bug : 'a tree -> 'a tree = <fun>


val avl_insert_bug : 'a -> 'a tree -> 'a tree = <fun>


val build : ('a -> 'b tree -> 'b tree) -> 'a list -> 'b tree = <fun>


val good : ('a -> 'a tree -> 'a tree) -> 'a list -> bool = <fun>


1 2 3 insert 3, 1, 2 (AVL) 1 2 3 insert 3, 1, 2 (rebalance_bug)

is the result a balanced BST holding the right values? case input expected avl_insert avl_insert_bug left-left [3; 2; 1] true ✓ true ✓ true right-right [1; 2; 3] true ✓ true ✓ true left-right [3; 1; 2] true ✓ true ✗ false right-left [1; 3; 2] true ✓ true ✗ false 1..15 in order [1; 2; 3; 4; 5; 6; 7; 8; 9; 10; 11; 12; 13; 14; 15] true ✓ true ✓ true 15..1 in reverse [15; 14; 13; 12; 11; 10; 9; 8; 7; 6; 5; 4; 3; 2; 1] true ✓ true ✓ true zig-zag [5; 1; 9; 2; 8; 3; 7; 4; 6] true ✓ true ✓ true passed 7 / 7 5 / 7

## 8. Test with hundreds of random inputs, and shrink the failure

Hand-picked fixtures find the bugs you thought of. A **property test** generates random inputs and checks a statement that should always be true. When it fails, it shrinks the failing input by dropping elements while the failure remains, which leaves a counterexample small enough to read.

In [126]:
let rand_list n range_ = List.init n (fun _ -> Random.int range_)

(* the smallest list we can reach by dropping elements while the property still fails *)
let rec shrink prop l =
  let rec try_drop pre = function
    | [] -> None
    | x :: rest -> let cand = List.rev_append pre rest in
        if not (prop cand) then Some cand else try_drop (x :: pre) rest
  in
  match try_drop [] l with Some smaller -> shrink prop smaller | None -> l

let check name ?(trials = 500) prop =
  Random.init 2026;
  let rec go i =
    if i > trials then [ name; Printf.sprintf "passed %d random trials" trials; "" ]
    else begin
      let l = rand_list (Random.int 12) 10 in
      if prop l then go (i + 1)
      else [ name; Printf.sprintf "FAILED on trial %d" i; "smallest failing input: " ^ show_list (shrink prop l) ]
    end
  in
  go 1

val rand_list : int -> int -> int list = <fun>


val shrink : ('a list -> bool) -> 'a list -> 'a list = <fun>


val check : string -> ?trials:int -> (int list -> bool) -> string list =
  <fun>


In [127]:
let () =
  let remove_prop rm l = let t = bst_of_list l in List.for_all (fun x -> remove_ok rm (x, t)) (0 :: l) in
  show_table ~title:"properties of the binary search tree"
    [ "property"; "result"; "counterexample" ]
    [ check "inorder (bst_of_list l) = sorted l without duplicates" (fun l -> inorder (bst_of_list l) = List.sort_uniq compare l);
      check "bst_of_list l is a BST" (fun l -> is_bst (bst_of_list l));
      check "mem x agrees with List.mem" (fun l -> let t = bst_of_list l in List.for_all (fun x -> mem x t = List.mem x l) (range 0 10));
      check "remove keeps a BST with the right values" (remove_prop remove);
      check "remove_bug keeps a BST with the right values" (remove_prop remove_bug);
      check "insert_bug builds a BST" (fun l -> is_bst (List.fold_left (fun t x -> insert_bug x t) Leaf l));
      check "AVL: balanced, a BST, right values" (good avl_insert);
      check "AVL with rebalance_bug: balanced, a BST, right values" (good avl_insert_bug) ]

property,result,counterexample
inorder (bst_of_list l) = sorted l without duplicates,passed 500 random trials,
bst_of_list l is a BST,passed 500 random trials,
mem x agrees with List.mem,passed 500 random trials,
remove keeps a BST with the right values,passed 500 random trials,
remove_bug keeps a BST with the right values,FAILED on trial 1,smallest failing input: [2; 1; 6]
insert_bug builds a BST,FAILED on trial 1,smallest failing input: [7; 6]
"AVL: balanced, a BST, right values",passed 500 random trials,
"AVL with rebalance_bug: balanced, a BST, right values",FAILED on trial 4,smallest failing input: [0; 7; 8; 6; 2; 1]


Read the counterexamples. `remove_bug` is caught by the smallest possible input that reaches its two-children branch. `rebalance_bug` is caught by a short list that contains a left-right or right-left shape. The shrunk list is only a local minimum, because shrinking here only drops elements; by hand, `[3; 1; 2]` is the smallest. A failure you can read in one line is a bug you can fix in a minute.

## What to change live

| Change this | In | Watch |
|---|---|---|
| base case `0` to `1` | `height` (section 2) | every row is off by one, even `empty` |
| `max` to `min` | `height` | perfect trees still pass |
| swap `preorder`'s operands | section 3 | the numbered badges move |
| `acc` `1` to `0` | `tree_fold` product (section 4) | every row fails |
| `<` to `>` in `insert` | section 6 | the picture becomes a mirror image |
| delete the two-children branch of `remove` | section 6 | subtrees vanish from the picture |
| remove one rotation case | `rebalance` (section 7) | `avl_insert` fails the left-right case |
| insert `1..n` in order | `bst_of_list` | height *n*, and the lookup time table |

## Exercises

1. Write `depth_of x t`, the number of nodes on the path from the root to `x`, and make a table that tests it. Which fixture trees need a case to catch an off-by-one?
2. Write `count_if p t` with `tree_fold`. What is the accumulator, and what is wrong with any other choice?
3. Write `is_balanced` so that it computes each height **once**, returning a pair or an option. How does the cost change?
4. `remove` replaces a two-child node with the smallest value in the right subtree. Change it to use the *largest value in the left subtree*. Which tests still pass, and what does the picture look like?
5. Write `level_of_each x t` that returns every value with its depth, and draw the tree with the depth as the badge.
6. Add a property: `height (avl_of_list l)` is at most `1.45 * log2 (n + 2)`. Run it against `bst_of_list` and watch it fail.

In [128]:
Random.init 9021;; Random.int 1000;;

- : unit = ()


- : int = 941


In [130]:
Random.init 3200;; Random.int 1000;;

- : unit = ()


- : int = 721
